In [ ]:
!pip install -q --upgrade pyarrow # Environment errors

In [1]:
import gc
import os
from datetime import date

import papermill as pm
import psutil
from dateutil.relativedelta import relativedelta
from openhexa.sdk import current_run
# from slack_sdk import WebClient
# from slack_sdk.errors import SlackApiError
# from slack_sdk.webhook import WebhookClient

import pyarrow ##

In [2]:
# Parameters

ANNEE = 2026
UPLOAD = True

In [3]:
pnlp_code_path = '/home/hexa/workspace/pnlp-tdb-pipeline/'
outputs_dir = 'papermill-outputs/'

In [ ]:
# last_month_date = date.today() + relativedelta(months=-1)

# if len(ANNEES) == 1 & (ANNEES[0] == last_month_date.year):
#     data_time = last_month_date.strftime("%b %Y")
    
#     if (date.today().day < 10):
#         verb = 'added'
#     else:
#         verb = 'updated'
        
#     run_notes = f"{data_time} data {verb}"
# else:
#     data_time = ', '.join([str(x) for x in ANNEES])
    
#     run_notes = f'Processed years {data_time}'

# run_notes = 'Aug 2023 data added'

# print(run_notes)

### Mise à jour des métadonnées des unités organisationnelles

In [ ]:
update_notebook = '00a_update_org_unit_metadata'

try:
    current_run.log_info("Updating DHIS2 organisation units metadata")
except AttributeError:
    print('not connected to pipeline run')


nb = pm.execute_notebook(
            input_path = f'{pnlp_code_path}{update_notebook}.ipynb',
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{update_notebook}_{ANNEE}.ipynb',
            parameters = dict(YEAR=ANNEE)
        )

try:
    current_run.log_info(f"DHIS2 organisation unit groups metadata ready ({ANNEE})")
except AttributeError:
    print('not connected to pipeline run')
try:
    current_run.log_info(f"DHIS2 organisation unit metadata ready ({ANNEE})")
except AttributeError:
    print('not connected to pipeline run')

print(f"Year: {ANNEE}")
print(f"Duration: {round(nb['metadata']['papermill']['duration'] / 60, 1)} min")
print(f"Exceptions: {nb['metadata']['papermill']['exception']}")

### Mise à jour de la population

In [ ]:
update_notebook = '00b_update_tdb_population'

if current_run:
    current_run.log_info(f"Updating DHIS2 population for year {ANNEE}")

nb = pm.execute_notebook(
            input_path = f'{pnlp_code_path}{update_notebook}.ipynb',
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{update_notebook}_{ANNEE}.ipynb',
            parameters = dict(ANNEE_A_ANALYSER=ANNEE)
        )

print(f"Year: {ANNEE}")
print(f"Duration: {round(nb['metadata']['papermill']['duration'] / 60, 1)} min")
print(f"Exceptions: {nb['metadata']['papermill']['exception']}")

### Préparer Extracts

In [ ]:
prep_notebook = '01_prep_DHIS_extracts'

try:
    current_run.log_info("Preparing DHIS2 extracts")
except AttributeError:
    print('not connected to pipeline run')


nb = pm.execute_notebook(
            input_path = f'{pnlp_code_path}{prep_notebook}.ipynb',
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{prep_notebook}_{ANNEE}.ipynb',
            parameters = dict(YEAR=ANNEE)
        )

print(f"Year: {ANNEE}")
print(f"Duration: {round(nb['metadata']['papermill']['duration'] / 60, 1)} min")
print(f"Exceptions: {nb['metadata']['papermill']['exception']}")

In [ ]:
del nb

gc.collect()

In [ ]:
#  # Getting % usage of virtual_memory ( 3rd field)
print('RAM memory % used:', psutil.virtual_memory()[2])
# # Getting usage of virtual_memory in GB ( 4th field)
print('RAM Used (GB):', psutil.virtual_memory()[3]/1000000000)

### PNLP Exhaustivité

In [ ]:
exhaustivite_notebook = '02_PNLP_exhaustivity'

try:
    current_run.log_info("Computing data element exhaustivity")
except AttributeError:
    print('not connected to pipeline run')

nb = pm.execute_notebook(
            input_path = f'{pnlp_code_path}{exhaustivite_notebook}.ipynb',
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{exhaustivite_notebook}_{ANNEE}.ipynb',
            parameters = dict(YEAR=ANNEE)
        )

print(f"Year: {ANNEE}")
print(f"Duration: {round(nb['metadata']['papermill']['duration'] / 60, 1)} min")
print(f"Exceptions: {nb['metadata']['papermill']['exception']}")

In [ ]:
del nb

In [ ]:
gc.collect()

In [ ]:
 # Getting % usage of virtual_memory ( 3rd field)
print('RAM memory % used:', psutil.virtual_memory()[2])
# Getting usage of virtual_memory in GB ( 4th field)
print('RAM Used (GB):', psutil.virtual_memory()[3]/1000000000)

### PNLP Indicators

In [ ]:
# indicator_notebook = "03a_indicator_computation"
indicator_notebook = "03a_indicator_computation_extra_indicators"


try:
    current_run.log_info("Computing routine indicators")
except AttributeError:
    print('not connected to pipeline run')


nb = pm.execute_notebook(
    input_path=f"{pnlp_code_path}{indicator_notebook}.ipynb",
    output_path=f"{pnlp_code_path}{outputs_dir}OUTPUT_{indicator_notebook}_{ANNEE}.ipynb",
    parameters=dict(ANNEE_A_ANALYSER=ANNEE, UPLOAD=UPLOAD),
    language="R",
)

print(f"ANNEE: {ANNEE}")
print(f"Duration: {round(nb['metadata']['papermill']['duration'] / 60, 1)} min")
print(f"Exceptions: {nb['metadata']['papermill']['exception']}")

In [ ]:
del nb

In [ ]:
gc.collect()

In [ ]:
 # Getting % usage of virtual_memory ( 3rd field)
print('RAM memory % used:', psutil.virtual_memory()[2])
# Getting usage of virtual_memory in GB ( 4th field)
print('RAM Used (GB):', psutil.virtual_memory()[3]/1000000000)

### PNLP Logistics

In [ ]:
# This version of logistics scripts removes the current year from the DB table
# and push the newly computed indicators (per year).
# It does not remove the table, only appends.
logistics_notebook = '03b_logistics_computation'

try:
    current_run.log_info("Computing logistics indicators")
except AttributeError:
    print('not connected to pipeline run')


nb = pm.execute_notebook(
            input_path = f'{pnlp_code_path}{logistics_notebook}.ipynb',
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{logistics_notebook}_{ANNEE}.ipynb',
            parameters = dict(ANNEE_A_ANALYSER=ANNEE,
                              UPLOAD=UPLOAD),
            request_save_on_cell_execute=False,
            language='R',
            timeout=4200
        )

print(f"Year: {ANNEE}")
print(f"Duration: {round(nb['metadata']['papermill']['duration'] / 60, 1)} min")
print(f"Exceptions: {nb['metadata']['papermill']['exception']}")

In [ ]:
del nb

In [ ]:
gc.collect()

In [ ]:
 # Getting % usage of virtual_memory ( 3rd field)
print('RAM memory % used:', psutil.virtual_memory()[2])
# Getting usage of virtual_memory in GB ( 4th field)
print('RAM Used (GB):', psutil.virtual_memory()[3]/1000000000)

### PNLP Exhaustivité (part 2)

In [ ]:
avail_notebook = '03c_exhaustivity_processing'

try:
    current_run.log_info("Formatting exhaustivity data")
except AttributeError:
    print('not connected to pipeline run')
        
    
nb = pm.execute_notebook(
            input_path = f'{pnlp_code_path}{avail_notebook}.ipynb',
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{avail_notebook}_{ANNEE}.ipynb',
            parameters = dict(ANNEE_A_ANALYSER=ANNEE,
                              UPLOAD=UPLOAD),
            language='R'
        )

print(f"Year: {ANNEE}")
print(f"Duration: {round(nb['metadata']['papermill']['duration'] / 60, 1)} min")
print(f"Exceptions: {nb['metadata']['papermill']['exception']}")

In [ ]:
del nb

In [ ]:
gc.collect()

In [ ]:
 # Getting % usage of virtual_memory ( 3rd field)
print('RAM memory % used:', psutil.virtual_memory()[2])
# Getting usage of virtual_memory in GB ( 4th field)
print('RAM Used (GB):', psutil.virtual_memory()[3]/1000000000)

### Superset aggregation scripts  

As we have moved the dashboard from Tableau to Superset, the data needs to be aggregated so Superset can handle it.  

> In case an update is required for the complete historical data (all years), run each notebook manually setting the corresponding 'year_targets' list.

In [ ]:
superset_code_path = "/home/hexa/workspace/pnlp-tdb-pipeline/superset_scripts/"

In [ ]:
update_notebook = 'availability_agg_update'

if current_run:
    current_run.log_info("Updating 'availability' aggregations table.")

nb = pm.execute_notebook(
            input_path = f'{superset_code_path}{update_notebook}.ipynb',            
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{update_notebook}_{ANNEE}.ipynb',
            parameters = dict(YEAR=ANNEE),
            cwd=superset_code_path,  # <-- run from the target's folder
        )

In [ ]:
update_notebook = 'indicators_nd_agg_nom_thematique_update'

if current_run:
    current_run.log_info("Updating 'indicators_nd' (nom thematique) aggregations table.")

nb = pm.execute_notebook(
            input_path = f'{superset_code_path}{update_notebook}.ipynb',            
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{update_notebook}_{ANNEE}.ipynb',
            parameters = dict(YEAR=ANNEE),
            cwd=superset_code_path,  # <-- run from the target's folder
        )

In [ ]:
update_notebook = 'indicators_nd_agg_proportion_update'

if current_run:
    current_run.log_info("Updating 'indicators_nd' (proportion) aggregations table.")

nb = pm.execute_notebook(
            input_path = f'{superset_code_path}{update_notebook}.ipynb',            
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{update_notebook}_{ANNEE}.ipynb',
            parameters = dict(YEAR=ANNEE),
            cwd=superset_code_path,  # <-- run from the target's folder
        )

In [ ]:
update_notebook = 'logistics_nd_agg_mois_stock_update'

if current_run:
    current_run.log_info("Updating 'logistics_nd' (mois stock) aggregations table.")

nb = pm.execute_notebook(
            input_path = f'{superset_code_path}{update_notebook}.ipynb',            
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{update_notebook}_{ANNEE}.ipynb',
            parameters = dict(YEAR=ANNEE),
            cwd=superset_code_path,  # <-- run from the target's folder    
        )

In [ ]:
update_notebook = 'logistics_nd_agg_rupture_update'

if current_run:
    current_run.log_info("Updating 'logistics_nd' (rupture) aggregations table.")

nb = pm.execute_notebook(
            input_path = f'{superset_code_path}{update_notebook}.ipynb',            
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{update_notebook}_{ANNEE}.ipynb',
            parameters = dict(YEAR=ANNEE),
            cwd=superset_code_path,  # <-- run from the target's folder  
        )

In [ ]:
update_notebook = 'logistics_nd_agg_stock_disp_update'

if current_run:
    current_run.log_info("Updating 'logistics_nd' (stock disp) aggregations table.")

nb = pm.execute_notebook(
            input_path = f'{superset_code_path}{update_notebook}.ipynb',            
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{update_notebook}_{ANNEE}.ipynb',
            parameters = dict(YEAR=ANNEE),
            cwd=superset_code_path,  # <-- run from the target's folder 
        )

In [ ]:
update_notebook = 'logistics_nd_table_produits_traceurs_update'

if current_run:
    current_run.log_info("Updating 'logistics_nd' (produits traceurs) table.")

nb = pm.execute_notebook(
            input_path = f'{superset_code_path}{update_notebook}.ipynb',            
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{update_notebook}_{ANNEE}.ipynb',
            parameters = dict(YEAR=ANNEE),
            cwd=superset_code_path,  # <-- run from the target's folder 
        )

### Fosa tables

In [ ]:
if current_run:
    current_run.log_info("Starting fosa level aggregation table updates...")

In [ ]:
update_notebook = 'availability_agg_update_fosa'

if current_run:
    current_run.log_info("Updating 'availablity' (FOSA) table.")

nb = pm.execute_notebook(
            input_path = f'{superset_code_path}{update_notebook}.ipynb',            
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{update_notebook}_{ANNEE}.ipynb',
            parameters = dict(YEAR=ANNEE),
            cwd=superset_code_path,  # <-- run from the target's folder 
        )

In [ ]:
update_notebook = 'indicators_nd_agg_proportion_update_fosa'

if current_run:
    current_run.log_info("Updating 'indicators_nd' (proportion FOSA) aggregations table.")

nb = pm.execute_notebook(
            input_path = f'{superset_code_path}{update_notebook}.ipynb',            
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{update_notebook}_{ANNEE}.ipynb',
            parameters = dict(YEAR=ANNEE),
            cwd=superset_code_path,  # <-- run from the target's folder 
        )

In [ ]:
update_notebook = 'logistics_nd_agg_rupture_update_fosa'

if current_run:
    current_run.log_info("Updating 'logistics_nd' (rupture FOSA) aggregations table.")

nb = pm.execute_notebook(
            input_path = f'{superset_code_path}{update_notebook}.ipynb',            
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{update_notebook}_{ANNEE}.ipynb',
            parameters = dict(YEAR=ANNEE),
            cwd=superset_code_path,  # <-- run from the target's folder 
    )

In [ ]:
update_notebook = 'logistics_nd_table_produits_traceurs_update_fosa'

if current_run:
    current_run.log_info("Updating 'logistics_nd' (produits traceurs FOSA) table.")

nb = pm.execute_notebook(
            input_path = f'{superset_code_path}{update_notebook}.ipynb',            
            output_path = f'{pnlp_code_path}{outputs_dir}OUTPUT_{update_notebook}_{ANNEE}.ipynb',
            parameters = dict(YEAR=ANNEE),
            cwd=superset_code_path,  # <-- run from the target's folder 
        )

### Notify the team via slack

In [ ]:
# message = (
#     "*DRC PNLP*\n"
#     f"     – *Years:* {ANNEE} \n"
#     f"     – *Notes:* {run_notes}"
# )

# flag = ("https://cdn-icons-png.flaticon.com/512/551/551848.png")

In [ ]:
# f = open(f"{pnlp_code_path}.slack_token", "r")
# slack_token = f.read().rstrip()
# f.close()

# f = open(f"{pnlp_code_path}.webhook", "r")
# webhook_url = f.read().rstrip()
# f.close()

In [ ]:
# webhook = WebhookClient(webhook_url)

# response = webhook.send(
#     text=message,
#     blocks=[
#         {
#             "type": "section",
#             "text": {
#                 "type": "mrkdwn",
#                 "text": ("A pipeline run has just finished"
#                          ":potable_water: \n"),
#             },
#         },
#         {"type": "divider"},
#         {
#             "type": "section",
#             "text": {"type": "mrkdwn", "text": message},
#             "accessory": {
#                 "type": "image",
#                 "image_url": flag,
#                 "alt_text": "DRC flag",
#             },
#         },
#         {"type": "divider"},
#     ],
# )
# assert response.status_code == 200
# assert response.body == "ok"